<a href="https://colab.research.google.com/github/wantuds-crypto/Ciencia-datos-curso/blob/main/Final_precios_inmuebles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Introducción y objetivo del análisis
### El problema
**Problemática:** predecir o explicar el precio de venta de una vivienda a partir de sus características físicas y de ubicación, para apoyar decisiones como fijar el precio de listado, detectar propiedades sub o sobrevaloradas y entender qué atributos elevan más el valor de una casa.

El precio de una vivienda depende de muchas cosas a la vez: su tamaño, su estado, su antigüedad, si tiene vista o frente al agua y, sobre todo, **dónde está**. Para quien compra, vende, construye o financia vivienda, entender **qué características se asocian con un precio más alto** permite tomar mejores decisiones: fijar precios de lista, priorizar zonas de inversión, decidir si conviene renovar o detectar ventas con precios anómalos.

### Objetivo general
Identificar, con los datos disponibles, qué características de la vivienda y de su ubicación se relacionan más con su **precio de venta**, y dejar el terreno preparado (datos limpios, variables construidas y evaluadas) para un futuro modelo de estimación de precios.

### Preguntas que queremos responder
1. ¿Qué tan confiables son los datos y qué problemas de calidad tienen?
2. ¿Cómo se distribuye el precio de venta? ¿Hay valores imposibles o sospechosos?
3. ¿Qué zonas y ciudades concentran las ventas y cuáles tienen precios más altos?
4. ¿Cómo cambia el precio según el tamaño, la antigüedad, el estado y la calidad de la vista?
5. ¿Qué variables (originales o construidas) tienen la relación más fuerte con el precio?
6. ¿Qué recomendaciones prácticas y qué próximos pasos se derivan?
\

In [213]:
# --- Librerías ---
from pathlib import Path
import re
import warnings
import os
import pandas as pd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import duckdb
from scipy import stats
import kagglehub

In [214]:
# Descarga del dataset
path = kagglehub.dataset_download("debayank2024/house-price-prediction")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'house-price-prediction' dataset.
Path to dataset files: /kaggle/input/house-price-prediction


## 1. Primer vistazo a los datos

Antes de limpiar o transformar nada, simplemente abrimos el archivo para ver que contiene: cuantas ventas hay, que columnas trae y como se ven los primeros registros. Este paso no cambia ni corrige nada, solo nos deja ver el punto de partida. El archivo trae 4.600 ventas de viviendas; cada fila es una casa vendida, con datos como precio, numero de habitaciones, tamano, ano de construccion y ubicacion. A partir de aqui decidiremos que hay que corregir.

In [215]:
df = pd.read_csv(os.path.join(path, "modified_data.csv"))
print("Filas y columnas:", df.shape)
df.info()
df.head()

Filas y columnas: (4600, 18)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   date            4600 non-null   object 
 1   price           4600 non-null   float64
 2   bedrooms        4600 non-null   float64
 3   bathrooms       4600 non-null   float64
 4   sqft_living     4600 non-null   int64  
 5   sqft_lot        4600 non-null   int64  
 6   floors          4600 non-null   float64
 7   waterfront      4600 non-null   int64  
 8   view            4600 non-null   int64  
 9   condition       4600 non-null   int64  
 10  sqft_above      4600 non-null   int64  
 11  sqft_basement   4600 non-null   int64  
 12  yr_built        4600 non-null   int64  
 13  yr_renovated    4600 non-null   int64  
 14  street          4600 non-null   object 
 15  city            4600 non-null   object 
 16  statezip        4600 non-null   object 
 17  pric

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,price_per_sqft
0,2014-05-02,"313,000.00",3.00,1.50,1340,7912,1.50,0,0,3,1340,0,1955,2005,18810 Densmore Ave N,Shoreline,WA 98133,233.58
1,2014-05-02,"2,384,000.00",5.00,2.50,3650,9050,2.00,0,4,5,3370,280,1921,0,709 W Blaine St,Seattle,WA 98119,653.15
2,2014-05-02,"342,000.00",3.00,2.00,1930,11947,1.00,0,0,4,1930,0,1966,0,26206-26214 143rd Ave SE,Kent,WA 98042,177.20
3,2014-05-02,"420,000.00",3.00,2.25,2000,8030,1.00,0,0,4,1000,1000,1963,0,857 170th Pl NE,Bellevue,WA 98008,210.00
4,2014-05-02,"550,000.00",4.00,2.50,1940,10500,1.00,0,0,4,1140,800,1976,1992,9105 170th Ave NE,Redmond,WA 98052,283.51


In [216]:
df.describe()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,price_per_sqft
count,"4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00","4,600.00"
mean,"551,962.99",3.40,2.16,"2,139.35","14,852.52",1.51,0.01,0.24,3.45,"1,827.27",312.08,"1,970.79",808.61,265.88
std,"563,834.70",0.91,0.78,963.21,"35,884.44",0.54,0.08,0.78,0.68,862.17,464.14,29.73,979.41,357.50
min,0.00,0.00,0.00,370.00,638.00,1.00,0.00,0.00,1.00,370.00,0.00,"1,900.00",0.00,0.00
25%,"322,875.00",3.00,1.75,"1,460.00","5,000.75",1.00,0.00,0.00,3.00,"1,190.00",0.00,"1,951.00",0.00,180.82
50%,"460,943.46",3.00,2.25,"1,980.00","7,683.00",1.50,0.00,0.00,3.00,"1,590.00",0.00,"1,976.00",0.00,243.86
75%,"654,962.50",4.00,2.50,"2,620.00","11,001.25",2.00,0.00,0.00,4.00,"2,300.00",610.00,"1,997.00","1,999.00",314.84
max,"26,590,000.00",9.00,8.00,"13,540.00","1,074,218.00",3.50,1.00,4.00,5.00,"9,410.00","4,820.00","2,014.00","2,014.00","22,533.90"


## 2. Limpieza de los datos

 Encontramos varios problemas "escondidos": ceros que no son ceros reales sino la forma en que el sistema marca "no aplica" o "no se registro", y algunos precios que son errores evidentes de digitacion. La regla que seguimos en todo el proceso es la misma: nunca eliminamos una fila completa; en cambio, marcamos el problema con una "bandera" y excluimos esos registros solo de los calculos donde afectarian el resultado. Asi, cualquier persona puede revisar despues que se marco y por que, y decidir si esta de acuerdo. A continuacion se resumen los cuatro grupos de problemas que se corrigieron.

#Formato Fechas
-pasar el date a formato fecha y además agregar una columna solo con el mes para determinar si hay meses que las viviendas son más costosas que otros.

In [217]:
df["date"] = pd.to_datetime(df["date"])
df = df.assign(
    mes=df['date'].dt.month
)

In [218]:
df.select_dtypes(include=['bool']).info()
df.select_dtypes(include=['object']).info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Empty DataFrame
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   street    4600 non-null   object
 1   city      4600 non-null   object
 2   statezip  4600 non-null   object
dtypes: object(3)
memory usage: 107.9+ KB


In [219]:
print("Nulos por columna:")
print(df.isnull().sum())

print("\nFilas duplicadas exactas:", df.duplicated().sum())

Nulos por columna:
date              0
price             0
bedrooms          0
bathrooms         0
sqft_living       0
sqft_lot          0
floors            0
waterfront        0
view              0
condition         0
sqft_above        0
sqft_basement     0
yr_built          0
yr_renovated      0
street            0
city              0
statezip          0
price_per_sqft    0
mes               0
dtype: int64

Filas duplicadas exactas: 0


In [244]:
import numpy as np
import re

MAPA_COLUMNAS = {
    "date": "fecha", "price": "precio", "bedrooms": "habitaciones", "bathrooms": "banos",
    "sqft_living": "area_construida_sqft", "sqft_lot": "area_lote_sqft", "floors": "pisos",
    "waterfront": "frente_agua", "view": "vista", "condition": "condicion",
    "sqft_above": "area_sobre_suelo_sqft", "sqft_basement": "area_sotano_sqft",
    "yr_built": "anio_construccion", "yr_renovated": "anio_renovacion", "street": "direccion",
    "city": "ciudad", "statezip": "estado_zip", "price_per_sqft": "precio_por_sqft",
}

def limpiar(raw: pd.DataFrame):
    df = raw.copy()
    bitacora = []

    # 1) Nombres, texto e identificador unico
    df = df.rename(columns=MAPA_COLUMNAS)
    df.columns = [re.sub(r"[^a-z0-9_]", "_", c.strip().lower()) for c in df.columns]
    for c in ["direccion", "ciudad", "estado_zip"]:
        df[c] = df[c].astype("string").str.strip().str.replace(r"\s{2,}", " ", regex=True)
    df["ciudad"] = df["ciudad"].str.title().str.replace("Seatac", "SeaTac")
    partes = df["estado_zip"].str.extract(r"^(?P<estado>[A-Z]{2})\s(?P<codigo_postal>\d{5})$")
    df = pd.concat([df.drop(columns="estado_zip"), partes], axis=1)
    df.insert(0, "id_inmueble", [f"INM-{i:05d}" for i in range(1, len(df) + 1)])
    bitacora.append(["1. Nombres, texto e identificador",
                      "Columnas en ingles sin id unico; texto con espacios/mayusculas inconsistentes",
                      "Traducir nombres, limpiar texto, separar estado/codigo postal, crear id_inmueble",
                      len(df)])

    # 2) Precio en cero y precios imposibles
    df["flag_precio_no_registrado"] = df["precio"] == 0
    df.loc[df["flag_precio_no_registrado"], "precio"] = np.nan
    df["precio_por_sqft"] = (df["precio"] / df["area_construida_sqft"]).round(2)
    log_pps = np.log(df["precio_por_sqft"].where(df["precio_por_sqft"] > 0))
    q1, q3 = log_pps.quantile([.25, .75]); iqr = q3 - q1
    df["flag_precio_extremo"] = ((log_pps < q1 - 3 * iqr) | (log_pps > q3 + 3 * iqr)).fillna(False)
    bitacora.append(["2. Precios en cero y precios imposibles",
                      f"{int(df['flag_precio_no_registrado'].sum())} ventas en US$0 y precios por pie2 absurdos",
                      "Marcar como nulo/atipico y excluir del analisis del precio (sin borrar filas)",
                      int(df["flag_precio_no_registrado"].sum() + df["flag_precio_extremo"].sum())])

    # 3) Habitaciones/banos en cero y renovacion inconsistente
    m0 = (df["habitaciones"] == 0) & (df["banos"] == 0)
    df.loc[m0, ["habitaciones", "banos"]] = np.nan
    m_nunca = df["anio_renovacion"] == 0
    m_inc = (df["anio_renovacion"] > 0) & (df["anio_renovacion"] < df["anio_construccion"])
    df["flag_renovacion_inconsistente"] = m_inc
    df.loc[m_nunca | m_inc, "anio_renovacion"] = np.nan
    bitacora.append(["3. Habitaciones/banos en 0 y renovacion inconsistente",
                      f"{int(m0.sum())} casas grandes con 0 habitaciones/banos; anos de renovacion en 0 o anteriores a la construccion",
                      "Convertir a nulo (dato desconocido / no aplica) y marcar inconsistencias",
                      int(m0.sum() + (m_nunca | m_inc).sum())])

    # 4) Direcciones repetidas y lotes atipicos
    df["flag_direccion_repetida"] = df.duplicated(["direccion", "ciudad", "codigo_postal"], keep=False)
    bitacora.append(["4. Direcciones repetidas y lotes muy grandes",
                      "Direcciones repetidas (casas distintas) y lotes rurales muy grandes",
                      "No se elimina nada: se marcan direcciones repetidas y se recomienda usar medianas para los lotes",
                      int(df["flag_direccion_repetida"].sum())])

    return df, pd.DataFrame(bitacora, columns=["Paso", "Problema encontrado", "Que se hizo", "Registros afectados"])

df, bitacora = limpiar(df_Path)   # usa el nombre que crea la celda de carga
bitacora

NameError: name 'df_Path' is not defined

**1. Nombres, texto e identificador unico.** Las columnas venian sin un identificador propio para cada vivienda, se limpiaron espacios y mayusculas en direccion y ciudad, se separo la columna que mezclaba estado y codigo postal en dos columnas distintas, y se creo un identificador (`id_inmueble`) para cada vivienda, porque la direccion se repite entre casas distintas (por ejemplo, conjuntos de apartamentos) y no sirve como identificador confiable por si sola.

**2. Precios en cero y precios imposibles.** 49 ventas (poco mas del 1%) tenian un precio de 0 dolares, lo cual no es un precio de mercado real; se marcaron y se convirtieron en "dato faltante", conservando la vivienda para el resto del analisis. Ademas, se detectaron 5 ventas con un precio por pie cuadrado absurdo, como una casa de 1.180 pies2 registrada en 26,6 millones de dolares, o ventas de apenas 7.800 y 84.350 dolares. Estos 5 casos se marcaron como "precio extremo" y se excluyen de cualquier analisis relacionado con el precio, aunque la vivienda sigue existiendo en la base de datos.

**3. Habitaciones y banos en cero, y renovaciones inconsistentes.** Dos viviendas grandes aparecian con 0 habitaciones y 0 banos, algo fisicamente poco creible; esos valores se convirtieron en "dato desconocido". Tambien encontramos que la columna de ano de renovacion usa el numero 0 para decir "nunca se ha renovado" (no que se renovo en el ano 0), y que 195 registros tenian una fecha de renovacion anterior a la fecha de construccion, lo cual es imposible. En ambos casos se corrigio el valor a "sin dato" y se dejo una marca para poder rastrear la decision.

**4. Direcciones repetidas y terrenos muy grandes.** Varias viviendas comparten la misma direccion base, pero corresponden a casas distintas (diferente tamano, habitaciones o ano de construccion), asi que no se eliminaron: solo se marcaron como "direccion repetida" para tenerlo presente. De forma similar, algunos terrenos son extremadamente grandes (lotes rurales), lo cual es real y no un error; se conservan, pero se recomienda usar la mediana en vez del promedio al resumir esta variable, porque el promedio se distorsiona con estos valores.

## 3. Casteo: darle a cada dato el tipo correcto
la fecha debe tratarse como fecha (para poder calcular antiguedad), un numero entero como numero entero, y un codigo postal debe guardarse como texto para no perder los ceros a la izquierda. Antes de este paso, por ejemplo, la fecha de venta llegaba como texto plano y no se podia usar para calcular meses o antiguedad. Este paso no cambia los valores, solo la forma en que se guardan.

In [221]:
# Casteo de tipos
df['date'] = pd.to_datetime(df['date'])
df['waterfront'] = df['waterfront'].astype(bool)

for c in ['city', 'street', 'statezip']:
    df[c] = df[c].astype('category')

cat_condition = pd.CategoricalDtype(categories=sorted(df['condition'].unique()), ordered=True)
df['condition'] = df['condition'].astype(cat_condition)

cat_view = pd.CategoricalDtype(categories=sorted(df['view'].unique()), ordered=True)
df['view'] = df['view'].astype(cat_view)

df.dtypes

,0
date,datetime64[ns]
price,float64
bedrooms,float64
bathrooms,float64
sqft_living,int64
sqft_lot,int64
floors,float64
waterfront,bool
view,category
condition,category


In [222]:
# Booleana: 2 estados que representan sí/no
df['waterfront'] = df['waterfront'].astype(bool)

# Nominal: categorías sin orden -> category simple
df['city'] = df['city'].astype('category')

# Ordinal: categorías CON orden -> category con ordered=True
cat_condition = pd.CategoricalDtype(categories=[1,2,3,4,5], ordered=True)
df['condition'] = df['condition'].astype(cat_condition)

###Analsis Inicial:
- Nominales: `street`, `city`, `statezip`
- Ordinales: `condition` , `view` , `floors`
- Booleanas: `waterfront`
- Numéricas(no nominal / ordinal / booleana):`price`, `sqft_living`, `sqft_lot`, `sqft_above`, `sqft_basement`, `price_per_sqft`,`bedrooms`, `bathrooms`, `yr_built`, `yr_renovated`,`yr_renovated`, `date`


In [245]:
CASTING = {
    "id_inmueble": "string", "fecha": "datetime", "precio": "float64",
    "habitaciones": "Int64", "banos": "float64", "area_construida_sqft": "Int64",
    "area_lote_sqft": "Int64", "pisos": "float64", "frente_agua": "boolean",
    "vista": "Int8", "condicion": "Int8", "area_sobre_suelo_sqft": "Int64",
    "area_sotano_sqft": "Int64", "anio_construccion": "Int16", "anio_renovacion": "Int16",
    "direccion": "string", "ciudad": "category", "estado": "category",
    "codigo_postal": "string", "precio_por_sqft": "float64",
}

for col, tipo in CASTING.items():
    df[col] = pd.to_datetime(df[col]) if tipo == "datetime" else df[col].astype(tipo)

df.dtypes.loc[list(CASTING)]

,0
id_inmueble,string[python]
fecha,datetime64[ns]
precio,float64
habitaciones,Int64
banos,float64
area_construida_sqft,Int64
area_lote_sqft,Int64
pisos,float64
frente_agua,boolean
vista,Int8


In [223]:
# Filas donde price o sqft_living son 0 o negativos
filas_invalidas = df[(df['price'] <= 0) | (df['sqft_living'] <= 0)]
print("Filas inválidas:", len(filas_invalidas))
df = df[(df['price'] > 0) & (df['sqft_living'] > 0)].reset_index(drop=True)

Filas inválidas: 49


In [224]:
# Agrupación geográfica de ciudades en zonas:
ZONAS = {
    "Seattle": ["Seattle"],
    "Eastside": ["Bellevue", "Redmond", "Kirkland", "Sammamish", "Issaquah", "Mercer Island", "Medina",
                 "Clyde Hill", "Yarrow Point", "Newcastle", "Woodinville", "Bothell", "Kenmore",
                 "Beaux Arts Village", "Inglewood-Finn Hill"],
    "Norte": ["Shoreline", "Lake Forest Park"],
    "Sur": ["Renton", "Kent", "Auburn", "Federal Way", "Burien", "Des Moines", "Tukwila", "SeaTac",
            "Normandy Park", "Covington", "Maple Valley", "Algona", "Pacific", "Milton", "Black Diamond", "Enumclaw"],
    "Rural / Cascadas": ["Snoqualmie", "North Bend", "Duvall", "Carnation", "Fall City", "Preston",
                         "Ravensdale", "Skykomish", "Snoqualmie Pass", "Vashon"],
}
CIUDAD_A_ZONA = {c: z for z, ciudades in ZONAS.items() for c in ciudades}


## Creamos llaves para poder hacer un análisis relacional real con JOINs sin inventar informacion.

| Tabla del modelo | | Llave primaria |,
|---|---|---|---|
| `inmuebles` | | `id_inmueble`
| `ventas` | | `id_venta` |
| `dim_ciudades` | | `ciudad` |
| `referencia_zip` | | `codigo_postal`|
              

## Paso 3 · Feature engineering: variables temporales.

## Caracteristicas de Variables para tener en cuenta en los features

| Columna | Tipo de dato | Descripción | Rol en el análisis | Observaciones de calidad |
|---|---|---|---|---|
| `date` | `datetime64` | Fecha de la venta | Fecha | Viene como texto; debe convertirse a fecha |
| `price` | `float (USD)` | Precio de venta en dólares | Variable objetivo (*target*) | Hay precios en 0 y valores extremos que no tienen sentido |
| `bedrooms` | `entero` | Número de habitaciones | Variable numérica | Viene como decimal; los valores 0 son poco creíbles |
| `bathrooms` | `decimal` | Número de baños; por ejemplo, 0.25 equivale a medio baño sin ducha | Variable numérica | Los valores 0 son poco creíbles |
| `sqft_living` | `entero` | Área habitable construida, en pies² | Variable numérica | Puede tener asimetría y valores muy altos |
| `sqft_lot` | `entero` | Área total del lote o terreno, en pies² | Variable numérica | Es muy asimétrica por lotes rurales o terrenos demasiado grandes |
| `floors` | `decimal` | Número de pisos; 1.5 representa un piso y medio | Variable numérica | Sin observaciones críticas iniciales |
| `waterfront` | `booleano` | Indica si la vivienda tiene frente al agua: 1 = sí, 0 = no | Variable categórica binaria | Variable muy desbalanceada: menos del 1 % tiene valor 1 |
| `view` | `entero ordinal (0–4)` | Calificación de la vista: 0 = sin vista destacada y 4 = excelente | Variable categórica ordinal | La mayoría de las viviendas tiene calificación 0 |
| `condition` | `entero ordinal (1–5)` | Estado de conservación: 1 = malo y 5 = excelente | Variable categórica ordinal | Hay pocas viviendas con calificaciones entre 1 y 2 |
| `sqft_above` | `entero` | Área construida sobre el nivel del suelo, en pies² | Variable numérica | Debe ser consistente con `sqft_living` y `sqft_basement` |
| `sqft_basement` | `entero` | Área de sótano, en pies²; 0 significa que no tiene sótano | Variable numérica | El 0 representa “no tiene sótano”, no un dato faltante |
| `yr_built` | `entero` | Año de construcción de la vivienda | Variable temporal | No debe ser mayor que el año de venta |
| `yr_renovated` | `entero / nulo` | Año de la última renovación; 0 significa que nunca se ha renovado | Variable temporal | El 0 es un valor centinela; no puede ser anterior al año de construcción |
| `street` | `texto` | Dirección de la vivienda: calle y número | Variable de texto | Puede repetirse, tener espacios innecesarios y no funciona como llave única |
| `city` | `categoría` | Ciudad donde se ubica la vivienda | Posible llave para JOIN | Puede tener diferencias de mayúsculas, espacios o categorías con pocos registros |
| `statezip` | `texto → estado + código postal` | Estado y código postal juntos; ejemplo: `WA 98133` |

In [225]:
df['ratio_bano_habitacion'] = df['bathrooms'] / df['bedrooms'].replace(0, np.nan)
df['ratio_bano_habitacion'] = df['ratio_bano_habitacion'].fillna(0)

df['tiene_sotano'] = df['sqft_basement'] > 0
df['ratio_living_lot'] = df['sqft_living'] / df['sqft_lot']
df['ratio_above_living'] = df['sqft_above'] / df['sqft_living']

df['precio_prom_ciudad'] = df.groupby('city', observed=True)['price'].transform('mean')
df['habitaciones_totales'] = df['bedrooms'] + df['bathrooms']

df['log_price'] = np.log1p(df['price'])
df['log_sqft_living'] = np.log1p(df['sqft_living'])

df.filter(regex='ratio_|sotano|prom_ciudad|log_|habitaciones_totales').head()

,ratio_bano_habitacion,tiene_sotano,ratio_living_lot,ratio_above_living,precio_prom_ciudad,habitaciones_totales,log_price,log_sqft_living
0,0.50,False,0.17,1.00,"420,392.36",4.50,12.65,7.20
1,0.50,True,0.40,0.92,"584,294.90",7.50,14.68,8.20
2,0.67,False,0.16,1.00,"441,880.99",5.00,12.74,7.57
3,0.75,True,0.25,0.50,"862,255.05",5.25,12.95,7.60
4,0.62,True,0.18,0.59,"667,649.53",6.50,13.22,7.57


In [226]:
#Variables categóricas
texto_cols = ["street", "city", "statezip"]
diag_texto = pd.DataFrame({
    "valores únicos": df[texto_cols].nunique(),
    "con espacios al inicio/fin": [(df[c] != df[c].str.strip()).sum() for c in texto_cols],
    "con espacios dobles": [df[c].str.contains(r"\s{2,}", regex=True).sum() for c in texto_cols],
    "únicos tras normalizar (minúsculas+trim)": [df[c].str.strip().str.lower().nunique() for c in texto_cols],
})
diag_texto["categorías duplicadas por formato"] = diag_texto["valores únicos"] - diag_texto["únicos tras normalizar (minúsculas+trim)"]
display(diag_texto)

patron_statezip = r"^[A-Z]{2} \d{5}$"
print("statezip con formato distinto de 'XX 99999':", (~df["statezip"].str.match(patron_statezip)).sum())
print("Estados distintos:", df["statezip"].str[:2].unique().tolist())
zip_multi_ciudad = df.assign(zip=df["statezip"].str[-5:]).groupby("zip")["city"].nunique()
print(f"Códigos postales asociados a más de una ciudad: {(zip_multi_ciudad > 1).sum()} de {zip_multi_ciudad.size}")

frec_ciudad = df["city"].value_counts()
print(f"\nCiudades: {frec_ciudad.size}. Ciudades con menos de {MIN_VENTAS_CIUDAD} ventas: {(frec_ciudad < MIN_VENTAS_CIUDAD).sum()}")
frec_ciudad.to_frame("ventas").T

,valores únicos,con espacios al inicio/fin,con espacios dobles,únicos tras normalizar (minúsculas+trim),categorías duplicadas por formato
street,4476,0,0,4476,0
city,44,0,0,44,0
statezip,77,0,0,77,0


statezip con formato distinto de 'XX 99999': 0
Estados distintos: ['WA']
Códigos postales asociados a más de una ciudad: 19 de 77

Ciudades: 44. Ciudades con menos de 30 ventas: 20


city,Seattle,Renton,Bellevue,Redmond,Kirkland,Issaquah,Kent,Auburn,Sammamish,Federal Way,Shoreline,Woodinville,Maple Valley,Mercer Island,Burien,Snoqualmie,Kenmore,Des Moines,North Bend,Covington,Duvall,Lake Forest Park,Newcastle,Bothell,Vashon,Tukwila,SeaTac,Enumclaw,Carnation,Normandy Park,Fall City,Medina,Clyde Hill,Black Diamond,Ravensdale,Pacific,Algona,Yarrow Point,Skykomish,Preston,Milton,Beaux Arts Village,Inglewood-Finn Hill,Snoqualmie Pass
ventas,1561,291,281,235,187,186,184,175,171,145,123,114,95,82,72,69,65,58,50,42,42,34,33,33,29,29,28,27,22,17,11,11,9,8,7,6,5,4,3,2,2,1,1,1


In [227]:
## Variables numericas
num_cols = df.select_dtypes("number").columns.tolist()
desc = df[num_cols].describe(percentiles=[.01, .25, .5, .75, .99]).T
desc["asimetría"] = df[num_cols].skew()
desc

,count,mean,std,min,1%,25%,50%,75%,99%,max,asimetría
price,"4,551.00","557,905.90","563,929.87","7,800.00","148,000.00","326,264.29","465,000.00","657,500.00","2,016,000.00","26,590,000.00",25.02
bedrooms,"4,551.00",3.39,0.90,0.00,2.00,3.00,3.00,4.00,6.00,9.00,0.47
bathrooms,"4,551.00",2.16,0.78,0.00,1.00,1.75,2.25,2.50,4.50,8.00,0.59
sqft_living,"4,551.00","2,132.37",955.95,370.00,720.00,"1,460.00","1,970.00","2,610.00","5,065.00","13,540.00",1.72
sqft_lot,"4,551.00","14,835.28","35,964.08",638.00,"1,024.50","5,000.00","7,680.00","10,978.00","204,949.50","1,074,218.00",11.33
floors,"4,551.00",1.51,0.54,1.00,1.00,1.00,1.50,2.00,3.00,3.50,0.55
sqft_above,"4,551.00","1,822.22",854.45,370.00,705.00,"1,190.00","1,590.00","2,300.00","4,460.00","9,410.00",1.45
sqft_basement,"4,551.00",310.15,461.99,0.00,0.00,0.00,0.00,600.00,"1,700.00","4,820.00",1.66
yr_built,"4,551.00","1,970.80",29.76,"1,900.00","1,903.00","1,951.00","1,976.00","1,997.00","2,014.00","2,014.00",-0.51
yr_renovated,"4,551.00",808.56,979.42,0.00,0.00,0.00,0.00,"1,999.00","2,014.00","2,014.00",0.39


In [228]:
#Rango de fechas
fechas = pd.to_datetime(df["date"], errors="coerce")
print(f"Fechas no convertibles: {fechas.isna().sum()}")
print(f"Rango de fechas: {fechas.min().date()} a {fechas.max().date()} ({(fechas.max() - fechas.min()).days} días)")
print(f"Días distintos con ventas: {fechas.dt.date.nunique()}")
fechas.dt.to_period("M").value_counts().sort_index().to_frame("ventas").T

Fechas no convertibles: 0
Rango de fechas: 2014-05-02 a 2014-07-10 (69 días)
Días distintos con ventas: 70


date,2014-05,2014-06,2014-07
ventas,1748,2155,648


In [229]:
anio_max_venta = fechas.dt.year.max()
reglas = {
    "Precio igual a 0": df["price"] == 0,
    "Habitaciones = 0": df["bedrooms"] == 0,
    "Baños = 0": df["bathrooms"] == 0,
    "Habitaciones no enteras": df["bedrooms"] % 1 != 0,
    "Año construcción posterior a la venta": df["yr_built"] > anio_max_venta,
    "Renovación antes de la construcción (y ≠ 0)": (df["yr_renovated"] > 0) & (df["yr_renovated"] < df["yr_built"]),
    "Renovación posterior a la venta": df["yr_renovated"] > anio_max_venta,
    "Sobre suelo + sótano ≠ área habitable": df["sqft_above"] + df["sqft_basement"] != df["sqft_living"],
    "Área construida > 3 × lote (inusual)": df["sqft_living"] > 3 * df["sqft_lot"],
    "price_per_sqft no coincide con price/sqft_living": (df["price"] / df["sqft_living"] - df["price_per_sqft"]).abs() > 0.01,
    "vista fuera de 0–4": ~df["view"].between(0, 4),
    "condición fuera de 1–5": ~df["condition"].between(1, 5),
}
diag_reglas = pd.DataFrame({"registros": {k: int(v.sum()) for k, v in reglas.items()}})
diag_reglas["%"] = (diag_reglas["registros"] / len(df) * 100).round(2)
diag_reglas

,registros,%
Precio igual a 0,0,0.00
Habitaciones = 0,2,0.04
Baños = 0,2,0.04
Habitaciones no enteras,0,0.00
Año construcción posterior a la venta,0,0.00
Renovación antes de la construcción (y ≠ 0),193,4.24
Renovación posterior a la venta,0,0.00
Sobre suelo + sótano ≠ área habitable,0,0.00
Área construida > 3 × lote (inusual),0,0.00
price_per_sqft no coincide con price/sqft_living,0,0.00


El dataset tiene calidad interna alta: No hay precios en cero, ni habitaciones o baños inválidos, ni errores en áreas, fechas de venta o variables ordinales. El único problema relevante son 193 registros (4.24%) donde el año de renovación es anterior al de construcción, lo cual es ilógico y debe corregirse o tratarse como dato faltante antes de modelar.|

In [230]:
#Posibles valores atípicos
def atipicos_tukey(s: pd.Series, k: float = 1.5, log: bool = False) -> pd.Series:
    """Devuelve una máscara booleana con los valores fuera de las vallas de Tukey."""
    x = np.log(s.where(s > 0)) if log else s
    q1, q3 = x.quantile([.25, .75])
    iqr = q3 - q1
    return (x < q1 - k * iqr) | (x > q3 + k * iqr)

K_TUKEY_EXTREMO = 3.0 # Definición de K_TUKEY_EXTREMO
cols_atip = ["price", "sqft_living", "sqft_lot", "bedrooms", "bathrooms", "price_per_sqft"]
diag_atip = pd.DataFrame({
    "inusuales (k=1,5)": [atipicos_tukey(df[c]).sum() for c in cols_atip],
    "extremos (k=3)": [atipicos_tukey(df[c], 3).sum() for c in cols_atip],
    "extremos en log (k=3)": [atipicos_tukey(df[c], 3, log=True).sum() for c in cols_atip],
    "máximo": [df[c].max() for c in cols_atip],
}, index=cols_atip)
display(diag_atip)

mask_ext = atipicos_tukey(df["price_per_sqft"], K_TUKEY_EXTREMO, log=True) & (df["price"] > 0)
print("Ventas con precio por pie² extremo (candidatas a error de captura):")
df.loc[mask_ext, ["date", "price", "sqft_living", "price_per_sqft", "bedrooms", "city"]].sort_values("price_per_sqft")

,"inusuales (k=1,5)",extremos (k=3),extremos en log (k=3),máximo
price,240,89,4,"26,590,000.00"
sqft_living,128,17,0,"13,540.00"
sqft_lot,540,400,79,"1,074,218.00"
bedrooms,115,3,37,9.00
bathrooms,130,18,1,8.00
price_per_sqft,136,13,5,"22,533.90"


Ventas con precio por pie² extremo (candidatas a error de captura):


,date,price,sqft_living,price_per_sqft,bedrooms,city
4351,2014-05-06,"7,800.00",780,10.00,2.00,Tukwila
4345,2014-06-01,"84,350.00",2630,32.07,4.00,Yarrow Point
4348,2014-05-05,"2,199,900.00",1120,"1,964.20",4.00,Covington
4346,2014-06-23,"12,899,000.00",2190,"5,889.95",3.00,Seattle
4350,2014-07-03,"26,590,000.00",1180,"22,533.90",3.00,Kent


Hay muchos valores atípicos, sobre todo en precio y área del lote. El precio máximo llega a 26.59 millones de USD y hay lotes de más de un millón de pies². Además, cinco ventas presentan precios por pie² extremos (muy bajos o muy altos), lo que sugiere errores de captura o casos excepcionales que deberían revisarse o eliminarse antes de modelar.



In [231]:
#Limpieza y trasnformacion final
MAPA_COLUMNAS = {
    "date": "fecha", "price": "precio", "bedrooms": "habitaciones", "bathrooms": "banos",
    "sqft_living": "area_construida_sqft", "sqft_lot": "area_lote_sqft", "floors": "pisos",
    "waterfront": "frente_agua", "view": "vista", "condition": "condicion",
    "sqft_above": "area_sobre_suelo_sqft", "sqft_basement": "area_sotano_sqft",
    "yr_built": "anio_construccion", "yr_renovated": "anio_renovacion", "street": "direccion",
    "city": "ciudad", "statezip": "estado_zip", "price_per_sqft": "precio_por_sqft",
}

def limpiar(raw: pd.DataFrame):
    """Aplica la limpieza documentada y devuelve (df_limpio, bitácora)."""
    df = raw.copy()
    log = []
    def registrar(paso, problema, decision, justificacion, afectados, riesgo):
        log.append({"Paso": paso, "Problema": problema, "Decisión": decision, "Justificación": justificacion,
                    "Registros afectados": int(afectados), "Riesgo / limitación": riesgo})

    # 1. Nombres de columnas
    df = df.rename(columns=MAPA_COLUMNAS)
    df.columns = [re.sub(r"[^a-z0-9_]", "_", c.strip().lower()) for c in df.columns]
    registrar(1, "Nombres en inglés y sin convención común", "Renombrar a español en snake_case",
              "Lectura más clara para el equipo y consultas SQL sin comillas", len(df), "Hay que mantener el mapa de nombres si cambia la fuente")

    # 2. Textos: trim, espacios dobles y formato consistente
    antes = df[["direccion", "ciudad", "estado_zip"]].copy()
    for c in ["direccion", "ciudad", "estado_zip"]:
        df[c] = df[c].astype("string").str.strip().str.replace(r"\s{2,}", " ", regex=True)
    df["ciudad"] = df["ciudad"].str.title().str.replace("Seatac", "SeaTac")
    df["estado_zip"] = df["estado_zip"].str.upper()
    cambiados = (antes.astype("string") != df[["direccion", "ciudad", "estado_zip"]]).any(axis=1).sum()
    registrar(2, "Posibles espacios sobrantes y mayúsculas inconsistentes", "Trim, colapsar espacios, ciudad en formato Título, estado en mayúsculas",
              "Evita categorías duplicadas ('Seattle' vs 'seattle ')", cambiados, "Control preventivo: en este archivo no cambió ningún valor")

    # 3. Separar estado y código postal
    partes = df["estado_zip"].str.extract(r"^(?P<estado>[A-Z]{2})\s(?P<codigo_postal>\d{5})$")
    df = pd.concat([df.drop(columns="estado_zip"), partes], axis=1)
    registrar(3, "Estado y código postal en una misma columna", "Separar en 'estado' y 'codigo_postal' (texto de 5 dígitos)",
              "El código postal es una llave de agregación útil; como texto conserva ceros a la izquierda", len(df), "El estado es constante (WA) y no aporta información")

    # 4. Identificador
    df.insert(0, "id_inmueble", [f"INM-{i:05d}" for i in range(1, len(df) + 1)])
    registrar(4, "La fuente no tiene llave primaria", "Crear id_inmueble secuencial (texto)",
              "Necesario para relacionar tablas; la dirección se repite y no sirve como llave", len(df), "El id depende del orden del archivo; no es estable entre versiones")

    # 5. Duplicados
    dup = df.drop(columns="id_inmueble").duplicated().sum()
    df["flag_direccion_repetida"] = df.duplicated(["direccion", "ciudad", "codigo_postal"], keep=False)
    registrar(5, "Duplicados exactos / direcciones repetidas", "No eliminar (0 duplicados exactos); marcar direcciones repetidas",
              "Las direcciones repetidas son viviendas distintas (áreas y habitaciones diferentes)", df["flag_direccion_repetida"].sum(),
              "Si alguna fuera doble registro, se contaría dos veces")

    # 6. Precio = 0 → nulo
    m = df["precio"] == 0
    df["flag_precio_no_registrado"] = m
    df.loc[m, "precio"] = np.nan
    registrar(6, "Precio igual a 0", "Convertir a nulo y marcar; conservar la vivienda",
              "Un precio 0 no es un precio de mercado; las características físicas siguen siendo válidas", m.sum(),
              "Estas viviendas quedan fuera de todo análisis del Target")

    # 7. Recalcular precio por pie²
    df["precio_por_sqft"] = (df["precio"] / df["area_construida_sqft"]).round(2)
    registrar(7, "precio_por_sqft hereda el precio 0", "Recalcular como precio / área construida",
              "Garantiza consistencia con el precio corregido", m.sum(), "Ninguno relevante")

    # 8. Precios extremos imposibles
    m_ext = atipicos_tukey(df["precio_por_sqft"], K_TUKEY_EXTREMO, log=True).fillna(False)
    df["flag_precio_extremo"] = m_ext
    registrar(8, "Precio por pie² imposible (valla de Tukey k=3 sobre el logaritmo)", "Marcar y excluir de los análisis del Target (no se borran)",
              "Valores como US$10/pie² o US$22.534/pie² son errores de captura evidentes; el resto de casas de lujo se conservan", m_ext.sum(),
              "Si alguno fuera una venta real atípica (p. ej. entre familiares), se pierde")

    # 9. Habitaciones y baños en 0
    m0 = (df["habitaciones"] == 0) & (df["banos"] == 0)
    df.loc[m0, ["habitaciones", "banos"]] = np.nan
    registrar(9, "Casas de más de 3.000 pie² con 0 habitaciones y 0 baños", "Convertir ambos a nulo (dato desconocido)",
              "Físicamente improbable; con nulo no se distorsionan ratios por habitación", m0.sum(), "Se pierden 2 valores; la fila se conserva")

    # 10. Renovación: centinela 0 e inconsistencias
    m_nunca = df["anio_renovacion"] == 0
    m_inc = (df["anio_renovacion"] > 0) & (df["anio_renovacion"] < df["anio_construccion"])
    df["flag_renovacion_inconsistente"] = m_inc
    df.loc[m_nunca | m_inc, "anio_renovacion"] = np.nan
    registrar(10, "0 = nunca renovada / renovación anterior a la construcción",
              "Convertir 0 a nulo ('no aplica'); año de renovación inconsistente a nulo y marcar",
              "Evita promedios de años sin sentido y evita dar por renovada una casa sin evidencia", (m_nunca | m_inc).sum(),
              f"{m_inc.sum()} viviendas quedan como 'sin renovación conocida' aunque pudieron renovarse")

    # 11. Atípicos de área/lote: se conservan
    m_lote = atipicos_tukey(df["area_lote_sqft"], 3)
    registrar(11, "Lotes muy grandes (valores extremos)", "Conservar; usar medianas, logaritmos y ratios",
              "Son lotes rurales plausibles, no errores", m_lote.sum(), "Las medias de área de lote no son representativas")
    return df, pd.DataFrame(log)

df, bitacora = limpiar(df)
bitacora

,Paso,Problema,Decisión,Justificación,Registros afectados,Riesgo / limitación
0,1,Nombres en inglés y sin convención común,Renombrar a español en snake_case,Lectura más clara para el equipo y consultas S...,4551,Hay que mantener el mapa de nombres si cambia ...
1,2,Posibles espacios sobrantes y mayúsculas incon...,"Trim, colapsar espacios, ciudad en formato Tít...",Evita categorías duplicadas ('Seattle' vs 'sea...,0,Control preventivo: en este archivo no cambió ...
2,3,Estado y código postal en una misma columna,Separar en 'estado' y 'codigo_postal' (texto d...,El código postal es una llave de agregación út...,4551,El estado es constante (WA) y no aporta inform...
3,4,La fuente no tiene llave primaria,Crear id_inmueble secuencial (texto),Necesario para relacionar tablas; la dirección...,4551,El id depende del orden del archivo; no es est...
4,5,Duplicados exactos / direcciones repetidas,No eliminar (0 duplicados exactos); marcar dir...,Las direcciones repetidas son viviendas distin...,145,"Si alguna fuera doble registro, se contaría do..."
5,6,Precio igual a 0,Convertir a nulo y marcar; conservar la vivienda,Un precio 0 no es un precio de mercado; las ca...,0,Estas viviendas quedan fuera de todo análisis ...
6,7,precio_por_sqft hereda el precio 0,Recalcular como precio / área construida,Garantiza consistencia con el precio corregido,0,Ninguno relevante
7,8,Precio por pie² imposible (valla de Tukey k=3 ...,Marcar y excluir de los análisis del Target (n...,Valores como US$10/pie² o US$22.534/pie² son e...,5,Si alguno fuera una venta real atípica (p. ej....
8,9,Casas de más de 3.000 pie² con 0 habitaciones ...,Convertir ambos a nulo (dato desconocido),Físicamente improbable; con nulo no se distors...,2,Se pierden 2 valores; la fila se conserva
9,10,0 = nunca renovada / renovación anterior a la ...,Convertir 0 a nulo ('no aplica'); año de renov...,Evita promedios de años sin sentido y evita da...,2899,193 viviendas quedan como 'sin renovación cono...


##4.Casting y tipificacion d evariables

In [232]:
tipos_antes = df.dtypes.astype(str)

CASTING = {
    # columna: (tipo destino, transformación, justificación)
    "id_inmueble": ("string", "Texto", "Identificador: nunca se opera aritméticamente"),
    "fecha": ("datetime64[ns]", "pd.to_datetime(formato ISO)", "Permite extraer mes, día y calcular antigüedades"),
    "precio": ("float64", "Decimal con 2 cifras", "Monto en USD; admite nulos (precio no registrado)"),
    "habitaciones": ("Int64", "Decimal → entero nullable", "Las habitaciones se cuentan en unidades enteras"),
    "banos": ("float64", "Se mantiene decimal", "Los baños se miden en cuartos (0,25, 0,5, 0,75)"),
    "area_construida_sqft": ("Int64", "Entero nullable", "Área en pies² enteros"),
    "area_lote_sqft": ("Int64", "Entero nullable", "Área en pies² enteros"),
    "pisos": ("float64", "Se mantiene decimal", "Existen medios pisos (1,5)"),
    "frente_agua": ("boolean", "0/1 → booleano", "Variable sí/no"),
    "vista": ("Int8", "Entero ordinal pequeño", "Escala 0–4 ordenada; se conserva numérica para correlaciones"),
    "condicion": ("Int8", "Entero ordinal pequeño", "Escala 1–5 ordenada; se crea además su etiqueta categórica"),
    "area_sobre_suelo_sqft": ("Int64", "Entero nullable", "Área en pies² enteros"),
    "area_sotano_sqft": ("Int64", "Entero nullable", "Área en pies² enteros"),
    "anio_construccion": ("Int16", "Entero nullable", "Año"),
    "anio_renovacion": ("Int16", "Entero nullable (0 → nulo)", "Año; nulo = sin renovación conocida"),
    "direccion": ("string", "Texto", "Texto libre, no analítico"),
    "ciudad": ("category", "Texto → categoría", "Pocas categorías repetidas: ahorra memoria y ordena"),
    "estado": ("category", "Texto → categoría", "Categoría (constante WA)"),
    "codigo_postal": ("string", "Texto de 5 dígitos", "Identificador geográfico: conserva ceros a la izquierda"),
    "precio_por_sqft": ("float64", "Decimal con 2 cifras", "USD por pie²"),
}

def castear(df: pd.DataFrame, reglas=CASTING) -> pd.DataFrame:
    df = df.copy()
    for col, (tipo, _, _) in reglas.items():
        try:
            if tipo.startswith("datetime"):
                df[col] = pd.to_datetime(df[col], format="%Y-%m-%d", errors="raise")
            else:
                df[col] = df[col].astype(tipo)
        except (ValueError, TypeError) as e:
            raise TypeError(f"No se pudo convertir '{col}' a {tipo}: {e}") from e
    return df

df = castear(df)
tabla_casting = pd.DataFrame([
    {"Variable": c, "Tipo original (fuente)": str(raw[k].dtype) if (k := {v: kk for kk, v in MAPA_COLUMNAS.items()}.get(c)) in raw else ("—(creada)" if c != "estado" and c != "codigo_postal" else "str (dentro de statezip)"),
     "Tipo tras limpieza": tipos_antes[c], "Tipo final": str(df[c].dtype), "Transformación aplicada": t, "Justificación": j}
    for c, (_, t, j) in CASTING.items()
])
tabla_casting

NameError: name 'raw' is not defined

In [ ]:
def validar_limpieza(df: pd.DataFrame, n_original: int) -> pd.DataFrame:
    checks = {
        "No se perdieron filas": len(df) == n_original,
        "id_inmueble es único": df["id_inmueble"].is_unique,
        "No hay precios ≤ 0": not (df["precio"] <= 0).any(),
        "fecha es datetime": pd.api.types.is_datetime64_any_dtype(df["fecha"]),
        "codigo_postal tiene 5 dígitos": df["codigo_postal"].str.fullmatch(r"\d{5}").all(),
        "Renovación ≥ construcción (cuando existe)": (df["anio_renovacion"].isna() | (df["anio_renovacion"] >= df["anio_construccion"])).all(),
        "Sobre suelo + sótano = área habitable": (df["area_sobre_suelo_sqft"] + df["area_sotano_sqft"] == df["area_construida_sqft"]).all(),
        "vista en 0–4 y condición en 1–5": df["vista"].between(0, 4).all() and df["condicion"].between(1, 5).all(),
        "precio_por_sqft consistente": ((df["precio"] / df["area_construida_sqft"] - df["precio_por_sqft"]).abs().fillna(0) < 0.01).all(),
    }
    res = pd.DataFrame({"Validación": list(checks), "Resultado": ["OK" if v else "FALLA" for v in checks.values()]})
    if (res["Resultado"] == "FALLA").any():
        raise AssertionError(f"Validaciones fallidas:\n{res[res['Resultado'] == 'FALLA']}")
    return res

display(validar_limpieza(df, len(df)))
df["flag_venta_valida"] = df["precio"].notna() & ~df["flag_precio_extremo"]
print(f"Ventas válidas para analizar el Target: {df['flag_venta_valida'].sum():,} de {len(df):,} "
      f"({df['flag_venta_valida'].mean():.1%})")

##Lo reaalizado hasta ahora

## . Limpieza de datos

- Eliminamos duplicados exactos.
- Imputamos nulos (mediana para numéricas, moda para categóricas).
- Casteamos cada columna al tipo que le corresponde.
- Corregimos el valor centinela `yr_renovated == 0` (no es "año cero", es "nunca renovada").
- Hay Viviendas Precios =0.

### Preguntas y decisiones para el trabajo

- por que razon dejamos los datos nulos
- por que razon usamos la media
- por que razon eliminados 100 registros y asi

##5. Creación de variables limpias
Con los datos limpios y tipificados construimos 27 variables candidatas

In [ ]:
def crear_variables(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    anio_ref = int(d["fecha"].dt.year.max())
    validas = d["flag_venta_valida"]
    dias = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]

    # --- Tiempo ---
    d["anio_venta"] = d["fecha"].dt.year.astype("Int16")
    d["mes_venta"] = d["fecha"].dt.month.astype("Int8")
    d["dia_semana_venta"] = pd.Categorical(d["fecha"].dt.dayofweek.map(dict(enumerate(dias))), categories=dias, ordered=True)
    d["venta_fin_de_semana"] = d["fecha"].dt.dayofweek >= 5
    # --- Antigüedad y renovación ---
    d["antiguedad_anios"] = (anio_ref - d["anio_construccion"]).astype("Int16")
    d["rango_antiguedad"] = pd.cut(d["antiguedad_anios"], bins=[-1, 10, 25, 50, 75, 200],
                                   labels=["0–10 años", "11–25 años", "26–50 años", "51–75 años", "Más de 75 años"])
    d["fue_renovado"] = d["anio_renovacion"].notna()
    d["edad_efectiva_anios"] = (anio_ref - d["anio_renovacion"].fillna(d["anio_construccion"])).astype("Int16")
    d["casa_reciente"] = d["anio_construccion"] >= anio_ref - 15
    # --- Tamaño y distribución ---
    d["area_construida_m2"] = (d["area_construida_sqft"].astype(float) * SQFT_A_M2).round(1)
    d["segmento_tamano"] = pd.qcut(d["area_construida_sqft"].astype(float), 4,
                                   labels=["Pequeña (Q1)", "Mediana (Q2)", "Grande (Q3)", "Muy grande (Q4)"])
    d["tiene_sotano"] = d["area_sotano_sqft"] > 0
    d["pct_area_sotano"] = (d["area_sotano_sqft"] / d["area_construida_sqft"]).astype(float).round(3)
    d["ratio_construido_lote"] = (d["area_construida_sqft"] / d["area_lote_sqft"]).astype(float).round(3)
    d["log_area_construida"] = np.log1p(d["area_construida_sqft"].astype(float)).round(4)   # (versión anterior: log_sqft_living)
    d["ratio_sobre_suelo"] = (d["area_sobre_suelo_sqft"] / d["area_construida_sqft"]).astype(float).round(3)  # (versión anterior: ratio_above_living)
    d["total_ambientes"] = (d["habitaciones"].astype(float) + d["banos"])
    d["banos_por_habitacion"] = (d["banos"] / d["habitaciones"].astype(float).replace(0, np.nan)).round(2)
    d["m2_por_habitacion"] = (d["area_construida_m2"] / d["habitaciones"].astype(float).replace(0, np.nan)).round(1)
    # --- Calidad y ubicación ---
    d["tiene_vista"] = d["vista"] > 0
    d["ubicacion_premium"] = d["frente_agua"] | (d["vista"] >= 3)  # interacción: agua o vista muy buena
    d["condicion_etiqueta"] = pd.Categorical(d["condicion"].astype(int).map({1: "Mala", 2: "Regular", 3: "Promedio", 4: "Buena", 5: "Excelente"}),
                                             categories=["Mala", "Regular", "Promedio", "Buena", "Excelente"], ordered=True)
    d["zona"] = d["ciudad"].astype(str).map(CIUDAD_A_ZONA).fillna("Sin zona asignada").astype("category")
    top = d["ciudad"].value_counts().head(TOP_N_CIUDADES).index
    d["ciudad_agrupada"] = d["ciudad"].astype(str).where(d["ciudad"].isin(top), "Otras").astype("category")
    # --- Derivadas del Target (solo descriptivas) ---
    d["log_precio"] = np.log(d["precio"].where(validas))
    umbral = d.loc[validas, "precio"].quantile(PERCENTIL_PRECIO_ALTO)
    d["precio_alto"] = pd.Series(np.where(validas, (d["precio"] >= umbral).astype(int), np.nan), index=d.index).astype("Int8")
    d["banda_precio"] = pd.qcut(d["precio"].where(validas), 5, labels=["Q1 más bajo", "Q2", "Q3", "Q4", "Q5 más alto"])
    med_zip = d[validas].groupby("codigo_postal")["precio_por_sqft"].median()
    d["precio_relativo_zip"] = (d["precio_por_sqft"].where(validas) / d["codigo_postal"].map(med_zip)).round(3)
    # (versión anterior: precio_prom_ciudad). Incluye el precio de la propia vivienda → fuga de información
    d["precio_prom_ciudad"] = d["precio"].where(validas).groupby(d["ciudad"], observed=True).transform("mean").round(0)
    d.attrs["umbral_precio_alto"] = umbral
    d.attrs["anio_ref"] = anio_ref
    return d


def formato_usd(valor: float) -> str:
    """Formatea un número como cadena de valor en USD."""
    return f"US${valor:,.0f}"

df = crear_variables(df)
sin_zona = (df["zona"] == "Sin zona asignada").sum()
if sin_zona:
    print(f"ADVERTENCIA: {sin_zona} viviendas en ciudades sin zona asignada; actualice el diccionario ZONAS.")
print(f"Año de referencia: {df.attrs['anio_ref']} | Umbral de 'precio alto' (percentil {PERCENTIL_PRECIO_ALTO:.0%}): {formato_usd(df.attrs['umbral_precio_alto'])} "
      f"({formato_usd(df.attrs['umbral_precio_alto'])})")
df.filter(["id_inmueble", "precio", "mes_venta", "antiguedad_anios", "rango_antiguedad", "fue_renovado", "area_construida_m2",
           "segmento_tamano", "zona", "ubicacion_premium", "precio_alto"]).head()

| Variable creada       | Variables origen                            | Regla de construcción                                               | Tipo                                    | Hipótesis o utilidad analítica                                                             |
| --------------------- | ------------------------------------------- | ------------------------------------------------------------------- | --------------------------------------- | ------------------------------------------------------------------------------------------ |
| mes_venta             | fecha                                       | Mes de la fecha de venta                                            | Numérica (entero)                       | ¿Hay estaciones dentro de may–jul?                                                     |
| dia_semana_venta      | fecha                                       | Nombre del día de la semana                                         | Categórica ordenada                     | ¿El día de cierre se asocia con el precio?                                                 |
| venta_fin_de_semana   | fecha                                       | 1 si sábado o domingo                                               | Binaria                                 | Comportamiento de cierre inusual                                                           |
| anio_venta            | fecha                                       | Año de la fecha de venta                                            | Numérica                                | Control temporal; se espera que sea constante                                              |
| antiguedad_anios      | anio_construccion                           | Año de referencia − año de construcción                             | Numérica                                | Las casas nuevas podrían valer más                                                         |
| rango_antiguedad      | antiguedad_anios                            | Cortes 0–10, 11–25, 26–50, 51–75, >75                               | Categórica ordenada                     | Equivalente al rango de edad para viviendas                                                |
| fue_renovado          | anio_renovacion                             | 1 si tiene año de renovación válido                                 | Binaria                                 | La renovación podría añadir valor                                                          |
| edad_efectiva_anios   | anio_renovacion, anio_construccion          | Años desde la última renovación o, si no hay, desde la construcción | Numérica                                | Mide qué tan actualizada está la casa                                                      |
| casa_reciente         | anio_construccion                           | 1 si fue construida en los últimos 15 años                          | Binaria                                 | Prima por construcción nueva                                                               |
| area_construida_m2    | area_construida_sqft                        | pies² × 0,092903                                                    | Numérica                                | Misma información en unidad local, m²                                                      |
| segmento_tamano       | area_construida_sqft                        | Cuartiles del área construida                                       | Categórica ordenada                     | Segmentación comercial por tamaño                                                          |
| tiene_sotano          | area_sotano_sqft                            | 1 si área de sótano > 0                                             | Binaria                                 | Espacio adicional                                                                          |
| pct_area_sotano       | area_sotano_sqft, area_construida_sqft      | Sótano / área construida                                            | Numérica (proporción)                   | Área bajo tierra suele valer menos                                                         |
| ratio_construido_lote | area_construida_sqft, area_lote_sqft        | Área construida / área de lote                                      | Numérica (ratio)                        | Densidad: urbano compacto vs. suburbano                                                    |
| log_area_construida   | area_construida_sqft                        | ln(1 + área construida)                                             | Numérica (transformada)                 | Relación más lineal con log_precio                                                         |
| ratio_sobre_suelo     | area_sobre_suelo_sqft, area_construida_sqft | Área sobre suelo / área construida                                  | Numérica (proporción)                   | Es exactamente 1 − pct_area_sotano; se espera redundancia                                  |
| total_ambientes       | habitaciones, banos                         | Habitaciones + baños                                                | Numérica                                | Programa arquitectónico total                                                              |
| banos_por_habitacion  | banos, habitaciones                         | Baños / habitaciones                                                | Numérica (ratio)                        | Estándar de acabados y confort                                                             |
| m2_por_habitacion     | area_construida_m2, habitaciones            | m² / habitaciones                                                   | Numérica (ratio)                        | Amplitud de los espacios                                                                   |
| tiene_vista           | vista                                       | 1 si vista > 0                                                      | Binaria                                 | Simplificación de una variable muy concentrada en 0                                        |
| ubicacion_premium     | frente_agua, vista                          | 1 si frente al agua o vista ≥ 3                                     | Binaria                                 | Atributos de ubicación escasos y valorados                                                 |
| condicion_etiqueta    | condicion                                   | 1–5 → Mala, Regular, Buena, Muy buena, Excelente                    | Categórica ordenada                     | Lectura amigable del estado de la vivienda                                                 |
| zona                  | ciudad                                      | Agrupación geográfica de ciudades en 5 zonas                        | Categórica                              | Segmentación principal de ubicación                                                        |
| ciudad_agrupada       | ciudad                                      | Top TOP_N_CIUDADES ciudades por volumen; resto como “Otras”         | Categórica                              | Reduce categorías con muy pocas ventas                                                     |
| log_precio            | precio                                      | ln(precio), solo ventas válidas                                     | Numérica (target transformado)          | Reduce la asimetría del target                                                             |
| precio_alto           | precio                                      | 1 si precio ≥ percentil PERCENTIL_PRECIO_ALTO                       | Binaria (target auxiliar)               | Permite comparar tasas entre segmentos                                                     |
| banda_precio          | precio                                      | Quintiles del precio                                                | Categórica ordenada derivada del target | Segmentación descriptiva; causa fuga de información si se usa para predecir                |
| precio_prom_ciudad    | precio, ciudad                              | Promedio del precio de su ciudad                                    | Numérica derivada del target            | Nivel de precios de la ciudad; incluye la propia venta y puede generar fuga de información |
| precio_relativo_zip   | precio_por_sqft, codigo_postal              | Precio por pie² ÷ mediana de su código postal                       | Numérica derivada del target            | Detecta viviendas caras o baratas respecto a su zona                                       |

##6. Integración de datos con SQL (DuckDB) y JOINs

In [ ]:
COLS_INMUEBLES = ["id_inmueble", "direccion", "ciudad", "codigo_postal", "habitaciones", "banos", "area_construida_sqft",
                  "area_construida_m2", "area_lote_sqft", "pisos", "frente_agua", "vista", "condicion", "condicion_etiqueta",
                  "anio_construccion", "anio_renovacion", "antiguedad_anios", "fue_renovado", "edad_efectiva_anios",
                  "tiene_sotano", "ubicacion_premium", "segmento_tamano"]
inmuebles = df[COLS_INMUEBLES].copy()
for c in ["ciudad", "condicion_etiqueta", "segmento_tamano"]:
    inmuebles[c] = inmuebles[c].astype(str)   # texto plano para SQL

ventas = (df.loc[df["precio"].notna(), ["id_inmueble", "fecha", "precio", "precio_por_sqft", "flag_precio_extremo"]]
            .rename(columns={"flag_precio_extremo": "atipico_extremo"}).reset_index(drop=True))
ventas.insert(0, "id_venta", [f"VEN-{i:05d}" for i in range(1, len(ventas) + 1)])

dim_ciudades = (df.groupby("ciudad", observed=True).size().rename("n_inmuebles").reset_index()
                  .assign(ciudad=lambda t: t["ciudad"].astype(str), zona=lambda t: t["ciudad"].map(CIUDAD_A_ZONA)))

con = duckdb.connect(database=":memory:")
for nombre, tabla in {"inmuebles": inmuebles, "ventas": ventas, "dim_ciudades": dim_ciudades}.items():
    con.register(f"_{nombre}", tabla)
    con.execute(f"CREATE OR REPLACE TABLE {nombre} AS SELECT * FROM _{nombre}")
    con.unregister(f"_{nombre}")

# Ejecutar la consulta SQL usando DuckDB
res_sql = con.execute("""
SELECT 'inmuebles' AS tabla, COUNT(*) AS filas, COUNT(DISTINCT id_inmueble) AS llaves_unicas FROM inmuebles
UNION ALL SELECT 'ventas', COUNT(*), COUNT(DISTINCT id_venta) FROM ventas
UNION ALL SELECT 'dim_ciudades', COUNT(*), COUNT(DISTINCT ciudad) FROM dim_ciudades
""").fetchdf()
display(res_sql)


In [ ]:
### Exploración
q1a = con.execute("""
SELECT id_venta, id_inmueble, fecha, precio, precio_por_sqft
FROM ventas
WHERE atipico_extremo = TRUE
ORDER BY precio_por_sqft DESC
LIMIT 10;
""").fetchdf()
q1b = con.execute("""
SELECT id_venta, id_inmueble, fecha, precio, precio_por_sqft
FROM ventas
WHERE atipico_extremo = FALSE
ORDER BY precio_por_sqft DESC
LIMIT 10;
""").fetchdf()
print("A) Ventas marcadas como atípicas extremas:"); display(q1a)
print("B) Ventas válidas con mayor precio por pie²:"); display(q1b)


In [233]:
q6 = sql("""
WITH base AS (
    SELECT a.id_venta, a.zona, a.codigo_postal, a.precio_por_sqft, a.ubicacion_premium,
           r.mediana_precio_sqft_zip
    FROM v_analitica AS a
    INNER JOIN referencia_zip AS r ON a.codigo_postal = r.codigo_postal
    WHERE a.atipico_extremo = FALSE
),
comparacion AS (
    SELECT *,
           precio_por_sqft / mediana_precio_sqft_zip AS precio_relativo,
           CASE
               WHEN precio_por_sqft / mediana_precio_sqft_zip >= 1.25 THEN 'Sobre el mercado (≥ +25 %)'
               WHEN precio_por_sqft / mediana_precio_sqft_zip <= 0.75 THEN 'Bajo el mercado (≤ −25 %)'
               ELSE 'En línea con el mercado'
           END AS posicion_mercado
    FROM base
)
SELECT zona,
       COUNT(*)                                                                         AS ventas,
       ROUND(100.0 * AVG(CASE WHEN posicion_mercado LIKE 'Sobre%' THEN 1 ELSE 0 END), 1) AS pct_sobre_mercado,
       ROUND(100.0 * AVG(CASE WHEN posicion_mercado LIKE 'Bajo%'  THEN 1 ELSE 0 END), 1) AS pct_bajo_mercado,
       ROUND(AVG(CASE WHEN ubicacion_premium THEN precio_relativo END), 2)              AS precio_rel_premium,
       ROUND(AVG(CASE WHEN NOT ubicacion_premium THEN precio_relativo END), 2)          AS precio_rel_no_premium
FROM comparacion
GROUP BY zona
ORDER BY pct_sobre_mercado DESC;
""")
q6

NameError: name 'sql' is not defined

El control confirma que SQL y Python producen rangos idénticos. La relación entre antigüedad y precio tiene forma de U: las viviendas de 11–25 y más de 75 años son más costosas. Aunque las antiguas son pequeñas, presentan mayor precio por pie², probablemente por su

In [ ]:
q6 = sql("""
WITH base AS (
    SELECT a.id_venta, a.zona, a.codigo_postal, a.precio_por_sqft, a.ubicacion_premium,
           r.mediana_precio_sqft_zip
    FROM v_analitica AS a
    INNER JOIN referencia_zip AS r ON a.codigo_postal = r.codigo_postal
    WHERE a.atipico_extremo = FALSE
),
comparacion AS (
    SELECT *,
           precio_por_sqft / mediana_precio_sqft_zip AS precio_relativo,
           CASE
               WHEN precio_por_sqft / mediana_precio_sqft_zip >= 1.25 THEN 'Sobre el mercado (≥ +25 %)'
               WHEN precio_por_sqft / mediana_precio_sqft_zip <= 0.75 THEN 'Bajo el mercado (≤ −25 %)'
               ELSE 'En línea con el mercado'
           END AS posicion_mercado
    FROM base
)
SELECT zona,
       COUNT(*)                                                                         AS ventas,
       ROUND(100.0 * AVG(CASE WHEN posicion_mercado LIKE 'Sobre%' THEN 1 ELSE 0 END), 1) AS pct_sobre_mercado,
       ROUND(100.0 * AVG(CASE WHEN posicion_mercado LIKE 'Bajo%'  THEN 1 ELSE 0 END), 1) AS pct_bajo_mercado,
       ROUND(AVG(CASE WHEN ubicacion_premium THEN precio_relativo END), 2)              AS precio_rel_premium,
       ROUND(AVG(CASE WHEN NOT ubicacion_premium THEN precio_relativo END), 2)          AS precio_rel_no_premium
FROM comparacion
GROUP BY zona
ORDER BY pct_sobre_mercado DESC;
""")
q6

In [234]:
#Rankings y funciones de ventana (`RANK`, `ROW_NUMBER`, `AVG OVER`, `PERCENT_RANK`)
q7 = sql("""
WITH ranking AS (
    SELECT
        zona, ciudad, id_venta, precio, area_construida_m2, ubicacion_premium,
        ROW_NUMBER()  OVER (PARTITION BY zona ORDER BY precio DESC)      AS fila_en_zona,
        RANK()        OVER (PARTITION BY zona ORDER BY precio DESC)      AS ranking_zona,
        ROUND(AVG(precio) OVER (PARTITION BY ciudad), 0)                 AS promedio_ciudad,
        ROUND(100 * PERCENT_RANK() OVER (ORDER BY precio), 1)            AS percentil_global
    FROM v_analitica
    WHERE atipico_extremo = FALSE
)
SELECT zona, ranking_zona, ciudad, id_venta, precio, area_construida_m2, ubicacion_premium,
       promedio_ciudad, ROUND(precio / promedio_ciudad, 1) AS veces_promedio_ciudad, percentil_global
FROM ranking
WHERE fila_en_zona <= 3
ORDER BY zona, ranking_zona;
""")
display(q7)

q7b = sql(f"""
SELECT zona, ciudad, cantidad_ventas, precio_mediano,
       RANK() OVER (PARTITION BY zona ORDER BY precio_mediano DESC) AS ranking_ciudad_en_zona
FROM (
    SELECT zona, ciudad, COUNT(*) AS cantidad_ventas, MEDIAN(precio) AS precio_mediano
    FROM v_analitica WHERE atipico_extremo = FALSE
    GROUP BY zona, ciudad HAVING COUNT(*) >= {MIN_VENTAS_CIUDAD}
) t
QUALIFY ranking_ciudad_en_zona <= 3
ORDER BY zona, ranking_ciudad_en_zona;
""")
q7b

NameError: name 'sql' is not defined

Con ROW_NUMBER, RANK y AVG OVER se identifica el top 3 de ventas por zona, comparando cada propiedad con el promedio de su ciudad. Bellevue destaca con US$7,06 M. La mayoría son propiedades premium o grandes. Los casos atípicos requieren verificación. Además, el ranking identifica las ciudades con mayores precios medianos, útil para inversión, comparables y auditorías

##7. Análisis exploratorio de datos (EDA) y visualizaciones

In [236]:
mediana_p, media_p = eda["precio"].median(), eda["precio"].mean()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
ax = axes[0]
ax.hist(eda["precio"], bins=np.arange(0, eda["precio"].quantile(.995) + 50_000, 50_000), color=AZUL, edgecolor=FONDO, linewidth=0.8)
ax.axvline(mediana_p, color=GRIS_TEXTO, lw=1.5, ls="--")
ax.axvline(media_p, color=NARANJA, lw=1.5)
ax.text(mediana_p, ax.get_ylim()[1] * .95, f"  Mediana {formato_usd(mediana_p)}", color=GRIS_TEXTO, fontsize=9)
ax.text(media_p, ax.get_ylim()[1] * .85, f"  Media {formato_usd(media_p)}", color=NARANJA, fontsize=9)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(formato_usd))
ax.set(xlabel="Precio de venta (USD, tramos de US$50 mil)", ylabel="Número de ventas", title="Escala original (hasta el percentil 99,5)")
ax = axes[1]
ax.hist(eda["log_precio"], bins=40, color=AZUL, edgecolor=FONDO, linewidth=0.8)
ticks = [1e5, 3e5, 1e6, 3e6]
ax.set_xticks(np.log(ticks)); ax.set_xticklabels([formato_usd(t) for t in ticks])
ax.set(xlabel="Precio de venta (escala logarítmica)", ylabel="Número de ventas", title="Escala logarítmica")
titular(fig, f"La mitad de las viviendas se vende entre {formato_usd(eda.precio.quantile(.25))} y {formato_usd(eda.precio.quantile(.75))}; unas pocas de lujo estiran el promedio",
        f"Distribución del precio de venta · n = {len(eda):,} ventas válidas · la media supera a la mediana por la cola de precios altos")
guardar(fig, "01_distribucion_precio"); plt.show()

resumen_target = pd.DataFrame({
    "Métrica": ["Ventas válidas", "Mínimo", "Percentil 25", "Mediana", "Media", "Percentil 75 (umbral precio alto)", "Percentil 99", "Máximo",
                "Asimetría (precio)", "Asimetría (log precio)", "% ventas de precio alto"],
    "Valor": [f"{len(eda):,}", formato_usd(eda.precio.min()), formato_usd(eda.precio.quantile(.25)), formato_usd(mediana_p), formato_usd(media_p),
              formato_usd(df.attrs["umbral_precio_alto"]), formato_usd(eda.precio.quantile(.99)), formato_usd(eda.precio.max()),
              f"{eda.precio.skew():.2f}", f"{eda.log_precio.skew():.2f}", f"{eda.precio_alto.mean():.1%}"]})
resumen_target

NameError: name 'eda' is not defined

La mediana es US 465 mil y la mitad de las ventas está entre US 326 mil y US 657 mil. La media (US 549 mil) aumenta por casas de lujo, generando asimetría de 4,0. Con logaritmo, baja a 0,31. Por ello, la mediana representa mejor el precio típico. Conviene usar log_precio como target.

In [ ]:
##preguntas y respuestas graficas

In [237]:
###  ¿Cómo se distribuyen las ventas por zona?
dist_zona = eda["zona"].value_counts().reindex(ORDEN_ZONAS[::-1]).to_frame("ventas")
dist_zona["%"] = dist_zona["ventas"] / dist_zona["ventas"].sum() * 100
orden_vol = dist_zona.sort_values("ventas").index
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(orden_vol, dist_zona.loc[orden_vol, "ventas"], color=[COLOR_ZONA[z] for z in orden_vol], height=0.6)
for i, z in enumerate(orden_vol):
    ax.text(dist_zona.loc[z, "ventas"] + 20, i, f"{dist_zona.loc[z, 'ventas']:,} ({dist_zona.loc[z, '%']:.1f} %)", va="center", fontsize=9, color=GRIS_TEXTO)
ax.set(xlabel="Número de ventas", ylabel="Zona"); ax.set_xlim(0, dist_zona["ventas"].max() * 1.22); ax.grid(axis="y", visible=False)
titular(fig, f"Seattle y Eastside concentran el {dist_zona.loc[['Seattle','Eastside'],'%'].sum():.0f} % de las ventas analizadas",
        "Número y porcentaje de ventas válidas por zona geográfica")
guardar(fig, "02_ventas_por_zona"); plt.show()
dist_zona.sort_values("ventas", ascending=False).round(1)

NameError: name 'eda' is not defined

In [ ]:
#¿Cómo se distribuyen las viviendas por rango de antigüedad?
dist_ant = eda["rango_antiguedad"].value_counts(sort=False).to_frame("ventas")
dist_ant["%"] = dist_ant["ventas"] / dist_ant["ventas"].sum() * 100
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(dist_ant.index.astype(str), dist_ant["%"], color=AZUL, width=0.6)
ax.bar_label(bars, labels=[f"{p:.1f} %\n({n:,})" for p, n in zip(dist_ant["%"], dist_ant["ventas"])], padding=3, fontsize=9, color=GRIS_TEXTO)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0)); ax.set_ylim(0, dist_ant["%"].max() * 1.25)
ax.set(xlabel="Antigüedad de la vivienda al momento de la venta", ylabel="% de ventas"); ax.grid(axis="x", visible=False)
top_rango = dist_ant["%"].idxmax()
titular(fig, f"El grupo más numeroso son las viviendas de {top_rango} ({dist_ant['%'].max():.0f} % de las ventas)",
        "Porcentaje de ventas válidas por rango de antigüedad (año de venta − año de construcción)")
guardar(fig, "03_ventas_por_antiguedad"); plt.show()

**Pregunta:** ¿cómo se reparte el parque de viviendas por antigüedad? *(Equivalente al rango de edad.)*
**Interpretación:** el grupo más numeroso son las casas de 26–50 años (27 %), seguido por las de 51–75 años (24 %). Las casas nuevas (0–10 años) son el 17 % y las de más de 75 años el 15 %.
**Conclusión:** se trata de un mercado mayoritariamente de **vivienda usada**; todos los rangos tienen suficiente volumen para compararlos.
**Acción:** la antigüedad es una buena variable de segmentación descriptiva, pero su relación con el precio debe analizarse controlando la zona (ver 10.6).

In [ ]:
#¿La antigüedad de las viviendas es distinta según la zona? *(sustituye a “género × rango de edad”)*
ct = pd.crosstab(eda["zona"], eda["rango_antiguedad"], normalize="index").reindex(ORDEN_ZONAS) * 100
fig, ax = plt.subplots(figsize=(11, 4.2))
sns.heatmap(ct, annot=True, fmt=".0f", cmap=sns.light_palette(AZUL, as_cmap=True), cbar_kws={"label": "% de las ventas de la zona"},
            linewidths=2, linecolor=FONDO, ax=ax, annot_kws={"fontsize": 10})
for t in ax.texts: t.set_text(t.get_text() + " %")
ax.set(xlabel="Rango de antigüedad", ylabel="Zona")
titular(fig, "Seattle tiene el parque de viviendas más antiguo; en Eastside y Rural predominan casas de menos de 50 años",
        "Distribución porcentual de la antigüedad dentro de cada zona (cada fila suma 100 %)")
guardar(fig, "04_zona_x_antiguedad"); plt.show()
chi2, p_chi, dof, _ = stats.chi2_contingency(pd.crosstab(eda["zona"], eda["rango_antiguedad"]))
n = len(eda); v_cramer = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
print(f"Prueba chi-cuadrado de independencia: chi2 = {chi2:,.0f}, p = {p_chi:.1e}; V de Cramér = {v_cramer:.2f}")

**Pregunta:** ¿la antigüedad de las casas es parecida en todas las zonas? *(Equivalente a género × rango de edad.)*
**Interpretación:** no. **Seattle** es la zona más antigua (38 % de sus ventas tienen más de 75 años; mediana 68 años). **Norte** está dominado por casas de 51–75 años (68 %). En **Eastside** y el **Sur** predominan las casas de 26–50 años, y **Rural/Cascadas** es la más nueva (60 % con 25 años o menos). La prueba chi-cuadrado confirma que la asociación es estadísticamente significativa y de intensidad moderada (V de Cramér = 0,31).
**Conclusión:** zona y antigüedad están entrelazadas. Cualquier diferencia de precio por antigüedad puede ser, en realidad, una diferencia por zona.
**Acción:** comparar antigüedades **dentro** de cada zona (tabla de la Sección 10.6).

In [ ]:
#¿Cómo se relaciona la zona con el precio?
por_zona = eda.groupby("zona", observed=True).agg(ventas=("precio", "size"), precio_mediano=("precio", "median"),
                                                   pct_precio_alto=("precio_alto", "mean"), precio_sqft_mediano=("precio_por_sqft", "median"))
por_zona["pct_precio_alto"] *= 100
por_zona = por_zona.sort_values("precio_mediano", ascending=False)
tasa_global = eda["precio_alto"].mean() * 100
fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
for ax, col, lab, fmt in [(axes[0], "precio_mediano", "Precio mediano (USD)", formato_usd), (axes[1], "pct_precio_alto", "% de ventas de precio alto", lambda v: f"{v:.0f} %")]:
    ax.barh(por_zona.index[::-1], por_zona[col][::-1], color=[COLOR_ZONA[z] for z in por_zona.index[::-1]], height=0.6)
    for i, v in enumerate(por_zona[col][::-1]):
        ax.text(v, i, "  " + fmt(v), va="center", fontsize=9, color=GRIS_TEXTO)
    ax.set_xlim(0, por_zona[col].max() * 1.3); ax.set(xlabel=lab, ylabel=""); ax.grid(axis="y", visible=False)
axes[0].xaxis.set_major_locator(mtick.MultipleLocator(200_000)); axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(formato_usd)); axes[0].set_ylabel("Zona")
axes[1].axvline(tasa_global, color=GRIS_TEXTO, ls="--", lw=1); axes[1].text(tasa_global, -0.7, f" promedio general {tasa_global:.0f} %", fontsize=8.5, color=GRIS_TEXTO)
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
z1 = por_zona.index[0]
titular(fig, f"{z1} es la zona más cara: {por_zona.loc[z1, 'pct_precio_alto']:.0f} % de sus ventas están en el 25 % de precios más altos",
        f"Precio mediano y tasa de 'precio alto' (≥ {formato_usd(df.attrs['umbral_precio_alto'])}) por zona")
guardar(fig, "05_zona_vs_target"); plt.show()
kw = stats.kruskal(*[g["precio"].values for _, g in eda.groupby("zona", observed=True)])
print(f"Kruskal-Wallis (¿las medianas de precio difieren entre zonas?): H = {kw.statistic:,.0f}, p = {kw.pvalue:.1e}")
por_zona.round(1)